# STEP 6 · 통계적 타당성 확인 (진단보다 먼저)

집단 차이가 우연이 아님을 검정으로 확인한다.
여기서도 **효과크기를 함께** 확인한다 (eta², Cramér's V).

- **ANOVA + eta²**: 3집단 이상 평균 비교 + 효과크기
- **카이제곱 + Cramér's V**: 범주형 연관 + 효과크기

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

import pandas as pd, numpy as np
from src.variable_mapping import MEDIA_LABELS
from src.stats_helpers import eta_squared, cramers_v
import warnings; warnings.filterwarnings('ignore')
dd = pd.read_csv(DATA_PROCESSED / "jtsi_2025.csv")

### 검정 1: ANOVA + eta² — 매체유형별 JTSI

In [2]:
groups = [dd[dd['SQ1']==c]['JTSI'].values for c in [1,2,3,4]]
F, p_anova, eta2 = eta_squared(groups)
print(f"ANOVA: F={F:.2f}, p={p_anova:.4f}, eta2={eta2:.4f}")
print(f"-> 차이는 유의(p<0.05)하나, eta2={eta2:.3f}로 효과크기는 작은~중간\n")
for c in [1,2,3,4]:
    sub = dd[dd['SQ1']==c]['JTSI']
    print(f"  {MEDIA_LABELS[c]}: {sub.mean():+.3f} (n={len(sub)})")

ANOVA: F=5.54, p=0.0009, eta2=0.0082
-> 차이는 유의(p<0.05)하나, eta2=0.008로 효과크기는 작은~중간

  신문사: +0.063 (n=1123)
  방송사: +0.120 (n=379)
  인터넷언론사: -0.441 (n=355)
  뉴스통신사: +0.245 (n=163)


**eta² 해석**: 0.01 작음 / 0.06 중간 / 0.14 큼. 매체유형이 JTSI 변동을 설명하는 비율.

### 검정 2: 카이제곱 + Cramér's V — 매체 × AI활용

In [3]:
ct = pd.crosstab(dd['SQ1'], dd['ai_user'])
chi2, p_chi, V = cramers_v(ct)
print(f"카이제곱: chi2={chi2:.2f}, p={p_chi:.4f}, Cramér's V={V:.4f}")
print("Cramér's V 해석: 0.1 작음 / 0.3 중간 / 0.5 큼")
ct.index = [MEDIA_LABELS[int(i)] for i in ct.index]
ct.columns = ['비활용','활용']; ct

카이제곱: chi2=17.23, p=0.0006, Cramér's V=0.0924
Cramér's V 해석: 0.1 작음 / 0.3 중간 / 0.5 큼


,비활용,활용
신문사,450,673
방송사,182,197
인터넷언론사,131,224
뉴스통신사,84,79


### 검정 결과 저장 (효과크기 컬럼 포함)

In [4]:
sig = pd.DataFrame([
    {'검정':'매체별 JTSI 차이','방법':'ANOVA','통계량':round(F,2),
     'p':round(p_anova,4),'효과크기':f'eta2={eta2:.3f}'},
    {'검정':'매체×AI활용 연관','방법':'카이제곱','통계량':round(chi2,2),
     'p':round(p_chi,4),'효과크기':f"V={V:.3f}"},
])
sig.to_csv(OUT_TABLES / "significance_tests.csv", index=False, encoding='utf-8-sig')
sig

,검정,방법,통계량,p,효과크기
0,매체별 JTSI 차이,ANOVA,5.54,0.0009,eta2=0.008
1,매체×AI활용 연관,카이제곱,17.23,0.0006,V=0.092


**수정 노트**: ANOVA/카이제곱 p값을 각각 다른 변수에 저장(버그 수정),
그리고 효과크기(eta², Cramér's V)를 추가했다. p값만으로는
"차이가 있다"까지만 말할 수 있고, 효과크기가 있어야 "얼마나 큰가"를 말한다.
